In [2]:
# Welcome to your new notebook
# Type here in the cell editor to add code!
from pyspark.sql.functions import col, isnull, when
from pyspark.sql.types import TimestampType
from datetime import date, timedelta

StatementMeta(, 8b406ec6-dc21-43c5-93d9-5cfb2e743100, 4, Finished, Available, Finished, False)

In [3]:
# remove this before adding to Data factory pipelines
start_date = date. today()-timedelta(7)

StatementMeta(, 8b406ec6-dc21-43c5-93d9-5cfb2e743100, 5, Finished, Available, Finished, False)

In [4]:
# load data to data frame
path = f"Files/{start_date}_earthquake_data.json"
df = spark.read.option("multiline","true").json(path)



StatementMeta(, 8b406ec6-dc21-43c5-93d9-5cfb2e743100, 6, Finished, Available, Finished, False)

In [5]:
df

StatementMeta(, 8b406ec6-dc21-43c5-93d9-5cfb2e743100, 7, Finished, Available, Finished, False)

DataFrame[geometry: struct<coordinates:array<double>,type:string>, id: string, properties: struct<alert:string,cdi:double,code:string,detail:string,dmin:double,felt:bigint,gap:bigint,ids:string,mag:double,magType:string,mmi:double,net:string,nst:bigint,place:string,rms:double,sig:bigint,sources:string,status:string,time:bigint,title:string,tsunami:bigint,type:string,types:string,tz:string,updated:bigint,url:string>, type: string]

In [6]:
df.head()

StatementMeta(, 8b406ec6-dc21-43c5-93d9-5cfb2e743100, 8, Finished, Available, Finished, False)

Row(geometry=Row(coordinates=[-76.289, -12.71, 61.2], type='Point'), id='us6000snu9', properties=Row(alert=None, cdi=4.1, code='6000snu9', detail='https://earthquake.usgs.gov/fdsnws/event/1/query?eventid=us6000snu9&format=geojson', dmin=0.899, felt=8, gap=137, ids=',us6000snu9,', mag=4.5, magType='mb', mmi=None, net='us', nst=40, place='19 km E of Coayllo, Peru', rms=0.78, sig=315, sources=',us,', status='reviewed', time=1775779068436, title='M 4.5 - 19 km E of Coayllo, Peru', tsunami=0, type='earthquake', types=',dyfi,origin,phase-data,', tz=None, updated=1775847968430, url='https://earthquake.usgs.gov/earthquakes/eventpage/us6000snu9'), type='Feature')

In [7]:
df = (
    df.select(
        'id',
        col('geometry.coordinates').getItem(0).alias('longitude'), 
        col('geometry.coordinates').getItem(1).alias('latitude'), 
        col('geometry.coordinates').getItem(2).alias('elevation'),
        col('properties.title').alias('title'),
        col('properties.place').alias('place_description'),
        col('properties.sig').alias('sig'),
        col('properties.mag').alias('mag'),
        col('properties.magType').alias('magType'),
        col('properties.time').alias('time'),
        col('properties.updated').alias("updated")
        )
    )

df.head()

StatementMeta(, 8b406ec6-dc21-43c5-93d9-5cfb2e743100, 9, Finished, Available, Finished, False)

Row(id='us6000snu9', longitude=-76.289, latitude=-12.71, elevation=61.2, title='M 4.5 - 19 km E of Coayllo, Peru', place_description='19 km E of Coayllo, Peru', sig=315, mag=4.5, magType='mb', time=1775779068436, updated=1775847968430)

In [8]:
#Handling NULL Values

df = (
    df
    .withColumn("longitude", when(isnull(col("longitude")), 0).otherwise(col("longitude")))
    .withColumn("latitude", when(isnull(col("latitude")), 0).otherwise(col("latitude")))
    .withColumn("time", when(isnull(col("time")), 0).otherwise(col("time")))
)

StatementMeta(, 8b406ec6-dc21-43c5-93d9-5cfb2e743100, 10, Finished, Available, Finished, False)

In [11]:

df = (
    df
    .withColumn("time", (col("time") / 1000).cast(TimestampType()))
    .withColumn("updated", (col("updated") / 1000).cast(TimestampType()))
)

StatementMeta(, 8b406ec6-dc21-43c5-93d9-5cfb2e743100, 13, Finished, Available, Finished, False)

In [13]:
#append the silver table

df.write.mode("append").saveAsTable("earthquake_events_silver")

StatementMeta(, 8b406ec6-dc21-43c5-93d9-5cfb2e743100, 15, Finished, Available, Finished, False)